# Session 3: IP Addressing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-03-ip-addressing.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-03_subnet_calc.py`

In [1]:
"""Session 3 - IPv4 subnet calculator, done by hand with bits AND checked
against Python's standard `ipaddress` module.

Run:  python session-03_subnet_calc.py
"""
import ipaddress


def ip_to_int(ip: str) -> int:
    a, b, c, d = (int(x) for x in ip.split("."))
    return (a << 24) | (b << 16) | (c << 8) | d


def int_to_ip(n: int) -> str:
    return ".".join(str((n >> s) & 0xFF) for s in (24, 16, 8, 0))


def ip_class(ip: str) -> str:
    first = int(ip.split(".")[0])
    if first < 128: return "A"
    if first < 192: return "B"
    if first < 224: return "C"
    if first < 240: return "D (multicast)"
    return "E (experimental)"


def manual(cidr: str) -> dict:
    """Compute subnet facts using only AND / OR / NOT on 32-bit integers."""
    ip, prefix = cidr.split("/")
    prefix = int(prefix)
    mask = (0xFFFFFFFF << (32 - prefix)) & 0xFFFFFFFF
    net = ip_to_int(ip) & mask                 # AND keeps the network bits
    bcast = net | (~mask & 0xFFFFFFFF)         # OR sets all host bits to 1
    hosts = max(2 ** (32 - prefix) - 2, 0) if prefix < 31 else 2 ** (32 - prefix)
    return {
        "mask": int_to_ip(mask),
        "mask_bits": f"{mask:032b}",
        "wildcard": int_to_ip(~mask & 0xFFFFFFFF),
        "network": int_to_ip(net),
        "broadcast": int_to_ip(bcast),
        "first": int_to_ip(net + 1) if prefix < 31 else int_to_ip(net),
        "last": int_to_ip(bcast - 1) if prefix < 31 else int_to_ip(bcast),
        "hosts": hosts,
    }


def report(cidr: str) -> None:
    m = manual(cidr)
    lib = ipaddress.ip_interface(cidr).network       # independent check
    assert m["network"] == str(lib.network_address)
    assert m["broadcast"] == str(lib.broadcast_address)
    ip = cidr.split("/")[0]
    addr = ipaddress.ip_address(ip)
    rfc1918 = [ipaddress.ip_network(n) for n in ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]
    docs = [ipaddress.ip_network(n) for n in ("192.0.2.0/24", "198.51.100.0/24", "203.0.113.0/24")]
    if any(addr in n for n in rfc1918): kind = "private (RFC 1918)"
    elif addr.is_loopback: kind = "loopback"
    elif any(addr in n for n in docs): kind = "documentation range (never routed)"
    else: kind = "public"
    print(f"\n{cidr}   class {ip_class(ip)}, {kind}")
    print(f"  mask      {m['mask']:<16} {m['mask_bits'][:8]}.{m['mask_bits'][8:16]}."
          f"{m['mask_bits'][16:24]}.{m['mask_bits'][24:]}")
    print(f"  wildcard  {m['wildcard']}")
    print(f"  network   {m['network']:<16} broadcast {m['broadcast']}")
    print(f"  hosts     {m['first']} - {m['last']}  ({m['hosts']} usable)")


def split(cidr: str, new_prefix: int) -> None:
    net = ipaddress.ip_network(cidr)
    subs = list(net.subnets(new_prefix=new_prefix))
    print(f"\nSplitting {cidr} into /{new_prefix}: {len(subs)} subnets "
          f"of {subs[0].num_addresses - 2} hosts each")
    for s in subs:
        h = list(s.hosts())
        print(f"  {str(s):<18} hosts {h[0]} - {h[-1]}  broadcast {s.broadcast_address}")


if __name__ == "__main__":
    for c in ["192.168.10.77/24", "172.16.45.200/20", "10.1.2.3/8", "192.168.1.130/26",
              "203.0.113.9/30", "127.0.0.1/8"]:
        report(c)
    split("192.168.50.0/24", 26)
    print("\nAll manual results matched the ipaddress module.")


192.168.10.77/24   class C, private (RFC 1918)
  mask      255.255.255.0    11111111.11111111.11111111.00000000
  wildcard  0.0.0.255
  network   192.168.10.0     broadcast 192.168.10.255
  hosts     192.168.10.1 - 192.168.10.254  (254 usable)

172.16.45.200/20   class B, private (RFC 1918)
  mask      255.255.240.0    11111111.11111111.11110000.00000000
  wildcard  0.0.15.255
  network   172.16.32.0      broadcast 172.16.47.255
  hosts     172.16.32.1 - 172.16.47.254  (4094 usable)

10.1.2.3/8   class A, private (RFC 1918)
  mask      255.0.0.0        11111111.00000000.00000000.00000000
  wildcard  0.255.255.255
  network   10.0.0.0         broadcast 10.255.255.255
  hosts     10.0.0.1 - 10.255.255.254  (16777214 usable)

192.168.1.130/26   class C, private (RFC 1918)
  mask      255.255.255.192  11111111.11111111.11111111.11000000
  wildcard  0.0.0.63
  network   192.168.1.128    broadcast 192.168.1.191
  hosts     192.168.1.129 - 192.168.1.190  (62 usable)

203.0.113.9/30   class C

## `session-03_ipv6.py`

In [2]:
"""Session 3 - IPv6 essentials: compress/expand, address types and EUI-64.

Run:  python session-03_ipv6.py
"""
import ipaddress


def eui64_interface_id(mac: str) -> str:
    """Build the 64-bit interface ID from a 48-bit MAC (modified EUI-64).
    1. split the MAC in half and insert FF:FE in the middle
    2. flip bit 7 (the universal/local bit) of the first byte
    """
    b = [int(x, 16) for x in mac.split(":")]
    b = b[:3] + [0xFF, 0xFE] + b[3:]
    b[0] ^= 0x02
    groups = [f"{b[i]:02x}{b[i + 1]:02x}" for i in range(0, 8, 2)]
    return ":".join(groups)


def classify(addr: str) -> str:
    a = ipaddress.ip_address(addr)
    if a.is_loopback: return "loopback (::1)"
    if a.is_link_local: return "link-local (fe80::/10, never routed)"
    if a.is_multicast: return "multicast (ff00::/8)"
    if a in ipaddress.ip_network("fc00::/7"): return "unique local (fc00::/7, like private IPv4)"
    if a in ipaddress.ip_network("2001:db8::/32"): return "documentation (2001:db8::/32)"
    if a in ipaddress.ip_network("2000::/3"): return "global unicast (2000::/3)"
    if a.is_unspecified: return "unspecified (::)"
    return "other"


if __name__ == "__main__":
    print("Compress / expand:")
    for full in ["2001:0db8:0000:0000:0000:ff00:0042:8329",
                 "fe80:0000:0000:0000:0202:b3ff:fe1e:8329",
                 "0000:0000:0000:0000:0000:0000:0000:0001"]:
        a = ipaddress.IPv6Address(full)
        print(f"  {a.exploded}  ->  {a.compressed}")

    print("\nAddress types:")
    for x in ["::1", "fe80::1", "ff02::1", "fd12:3456:789a::10",
              "2001:db8:acad:1::5", "2606:4700:4700::1111", "::"]:
        print(f"  {x:<24} {classify(x)}")

    mac = "00:1a:2b:3c:4d:5e"
    iid = eui64_interface_id(mac)
    prefix = ipaddress.IPv6Network("2001:db8:acad:10::/64")
    full = ipaddress.IPv6Address(f"2001:db8:acad:10:{iid}")
    print(f"\nEUI-64 from MAC {mac}: interface ID {iid}")
    print(f"  SLAAC address in {prefix}: {full}")
    print(f"  link-local address:            {ipaddress.IPv6Address('fe80::' + iid)}")

    print(f"\nA single /64 holds {prefix.num_addresses:,} addresses")
    print(f"IPv4 has only        {2**32:,} in total")

Compress / expand:
  2001:0db8:0000:0000:0000:ff00:0042:8329  ->  2001:db8::ff00:42:8329
  fe80:0000:0000:0000:0202:b3ff:fe1e:8329  ->  fe80::202:b3ff:fe1e:8329
  0000:0000:0000:0000:0000:0000:0000:0001  ->  ::1

Address types:
  ::1                      loopback (::1)
  fe80::1                  link-local (fe80::/10, never routed)
  ff02::1                  multicast (ff00::/8)
  fd12:3456:789a::10       unique local (fc00::/7, like private IPv4)
  2001:db8:acad:1::5       documentation (2001:db8::/32)
  2606:4700:4700::1111     global unicast (2000::/3)
  ::                       unspecified (::)

EUI-64 from MAC 00:1a:2b:3c:4d:5e: interface ID 021a:2bff:fe3c:4d5e
  SLAAC address in 2001:db8:acad:10::/64: 2001:db8:acad:10:21a:2bff:fe3c:4d5e
  link-local address:            fe80::21a:2bff:fe3c:4d5e

A single /64 holds 18,446,744,073,709,551,616 addresses
IPv4 has only        4,294,967,296 in total
